# Tugas Kelompok Simulasi Probabilitas dan Statistika (Member E)
**Mata Kuliah:** Statistika dan Probabilitas  
**File Identitas:** `simulation.ipynb` / `simulation.py`

---

## Deskripsi Tugas
Notebook ini berisi implementasi tiga komponen utama simulasi stokastik dan struktur data probabilistik:
1. **Metode Monte Carlo** untuk estimasi probabilitas umum.
2. **Bloom Filter** untuk pengecekan keanggotaan set yang efisien dengan perhitungan *False Positive Rate* (FPR).
3. **Markov Chain Monte Carlo (MCMC)** dengan pendekatan algoritma Metropolis-Hastings untuk menyelesaikan *Knapsack Problem*.

In [ ]:
import numpy as np
import random
import math
import matplotlib.pyplot as plt

## 1. Estimasi Probabilitas (Metode Monte Carlo)

Metode Monte Carlo adalah teknik komputasi yang memanfaatkan sampling acak berulang untuk memperoleh hasil numerik. Pendekatan ini sangat berguna untuk menghitung peluang dari suatu kejadian kompleks yang sulit diselesaikan secara analitis.

### Prinsip Kerja:
Fungsi `estimate_probability(event_fn, n_trials)` akan menjalankan fungsi kejadian (`event_fn`) sebanyak $n$ kali percobaan (`n_trials`). Probabilitas diestimasi dengan rumus:

$$\hat{P}(E) = \frac{\text{Jumlah Kejadian Sukses}}{\text{Total Percobaan (n\_trials)}}$$

Berdasarkan *Law of Large Numbers*, nilai estimasi ini akan semakin mendekati nilai probabilitas teoritis seiring bertambahnya jumlah $n$.

In [ ]:
def estimate_probability(event_fn, n_trials=50000):
    """
    Mengestimasi probabilitas suatu kejadian menggunakan metode Monte Carlo.
    
    :param event_fn: Fungsi yang mengembalikan True jika kejadian terjadi, False jika tidak.
    :param n_trials: Jumlah simulasi yang dijalankan.
    :return: Estimasi probabilitas (float).
    """
    successes = 0
    for _ in range(n_trials):
        if event_fn():
            successes += 1
            
    return successes / n_trials

# -----------------------------------------------------------------
# CONTOH PENGUJIAN: Menghitung peluang muncul angka GANJIL pada dadu (Teoritis: 0.5)
# -----------------------------------------------------------------
def cek_dadu_ganjil():
    dadu = random.randint(1, 6)
    return dadu % 2 != 0

peluang_estimasi = estimate_probability(cek_dadu_ganjil, n_trials=50000)
print(f"=== HASIL ESTIMASI PROBABILITAS ===")
print(f"Hasil Simulasi Monte Carlo : {peluang_estimasi}")
print(f"Nilai Teoritis             : 0.5")

## 2. Struktur Data Bloom Filter

**Bloom Filter** adalah struktur data ruang-efisien berbasis probabilitas yang digunakan untuk menguji apakah suatu elemen merupakan anggota dari suatu set. 

### Karakteristik Utama:
* **False Negative Tidak Mungkin Terjadi:** Jika filter mengembalikan nilai `False`, maka elemen tersebut *pasti* belum pernah dimasukkan.
* **False Positive Mungkin Terjadi:** Jika filter mengembalikan nilai `True`, elemen tersebut *mungkin* sudah dimasukkan, atau terjadi tabrakan (*hash collision*).

### Formulasi Teoritis False Positive Rate (FPR):
Jika kita memasukkan $n$ elemen ke dalam Bloom Filter berukuran $m$ bit menggunakan $k$ fungsi hash, maka probabilitas terjadinya *False Positive* secara teoritis dapat dihitung dengan rumus:

$$FPR = \left( 1 - e^{-\frac{k \cdot n}{m}} \right)^k$$

In [ ]:
class BloomFilter:
    def __init__(self, m, k):
        """
        Inisialisasi Bloom Filter.
        :param m: Ukuran bit array (size of bit array)
        :param k: Jumlah fungsi hash (number of hash functions)
        """
        self.m = m
        self.k = k
        self.bit_array = [0] * m

    def _hashes(self, item):
        """
        Fungsi internal untuk menghasilkan k indeks hash yang berbeda 
        menggunakan trik salting berbasis bawaan hash() Python.
        """
        indices = []
        for i in range(self.k):
            # Menggunakan string unique salt untuk setiap fungsi hash i
            hash_val = hash(f"{item}-{i}")
            indices.append(hash_val % self.m)
        return indices

    def add(self, item):
        """Menambahkan elemen ke dalam Bloom Filter."""
        for index in self._hashes(item):
            self.bit_array[index] = 1

    def contains(self, item):
        """
        Memeriksa apakah elemen *mungkin* ada di dalam Bloom Filter.
        """
        for index in self._hashes(item):
            if self.bit_array[index] == 0:
                return False  # Pasti belum pernah ditambahkan
        return True  # Mungkin sudah ditambahkan (bisa False Positive)

    def theoretical_fpr(self, n):
        """
        Menghitung nilai False Positive Rate (FPR) secara teoritis.
        Formula: (1 - e^(-k * n / m))^k
        :param n: Jumlah elemen yang telah dimasukkan ke dalam filter
        """
        if self.m == 0:
            return 1.0
        exponent = - (self.k * n) / self.m
        base = 1 - math.exp(exponent)
        return math.pow(base, self.k)

# -----------------------------------------------------------------
# CONTOH PENGUJIAN BLOOM FILTER
# -----------------------------------------------------------------
print(f"=== HASIL PENGUJIAN BLOOM FILTER ===")
bf = BloomFilter(m=1000, k=5)

# Tambahkan data audit contoh
data_input = ["transaksi_001", "transaksi_002", "transaksi_003"]
for data in data_input:
    bf.add(data)

# Cek data
print("Apakah 'transaksi_001' ada?", bf.contains("transaksi_001")) # True
print("Apakah 'transaksi_999' ada?", bf.contains("transaksi_999")) # False (atau True kecil jika FPR terjadi)
print(f"Teoritis FPR untuk {len(data_input)} elemen: {bf.theoretical_fpr(len(data_input)):.6f}")

## 3. Optimasi Knapsack Menggunakan MCMC (Metropolis-Hastings)

*Knapsack Problem* adalah masalah optimasi kombinatorial di mana kita harus memilih kombinasi barang yang memiliki total nilai (*value*) maksimum tanpa melebihi batas kapasitas berat (*capacity*) tertentu.

Karena ruang kombinasi barang bersifat eksponensial ($2^N$), kita menggunakan metode **Markov Chain Monte Carlo (MCMC)** dengan algoritma **Metropolis-Hastings** untuk menjelajahi ruang solusi secara cerdas.

### Logika Algoritma:
1. **Proposal State:** Pada setiap iterasi, satu barang dipilih secara acak untuk diubah statusnya (jika awalnya tidak diambil menjadi diambil, atau sebaliknya).
2. **Evaluasi Fungsi Tujuan:** Hitung total nilai ($V$) dan berat ($W$). Jika $W > \text{capacity}$, state diberikan penalti berat (nilai menjadi 0).
3. **Kriteria Penerimaan (Metropolis Acceptance):**
   * Jika $V_{\text{proposal}} > V_{\text{current}}$, proposal **selalu diterima**.
   * Jika $V_{\text{proposal}} \le V_{\text{current}}$, proposal diterima dengan probabilitas berbasis distribusi Boltzmann/eksponensial:
   
   $$\alpha = \exp(V_{\text{proposal}} - V_{\text{current}})$$

In [ ]:
def mcmc_knapsack(items, capacity, n_iter=100000):
    """
    Menyelesaikan Knapsack Problem menggunakan Algoritma Metropolis-Hastings (MCMC).
    
    :param items: List of dict, contoh: [{'weight': 10, 'value': 60}, ...]
    :param capacity: Kapasitas berat maksimum knapsack
    :param n_iter: Jumlah iterasi MCMC
    :return: (best_state, best_value)
    """
    num_items = len(items)
    
    # State awal: semua barang tidak diambil (0)
    current_state = np.zeros(num_items, dtype=int)
    current_value = 0
    current_weight = 0
    
    best_state = current_state.copy()
    best_value = current_value

    for _ in range(n_iter):
        # 1. Pilih satu item secara acak untuk di-flip statusnya (0->1 atau 1->0)
        proposal_idx = random.randint(0, num_items - 1)
        proposal_state = current_state.copy()
        proposal_state[proposal_idx] = 1 - proposal_state[proposal_idx]
        
        # 2. Hitung total berat dan nilai proposal
        proposal_weight = sum(proposal_state[i] * items[i]['weight'] for i in range(num_items))
        proposal_value = sum(proposal_state[i] * items[i]['value'] for i in range(num_items))
        
        # 3. Batasan kapasitas berat (Penalti keras jika melebihi kapasitas)
        if proposal_weight > capacity:
            proposal_value = 0 
            
        # 4. Kriteria Penerimaan Metropolis-Hastings
        if proposal_value > current_value:
            accept = True
        else:
            # Jika nilainya lebih buruk, masih ada peluang kecil untuk diterima demi eksplorasi area baru
            prob = math.exp(proposal_value - current_value) if proposal_value > 0 else 0
            accept = random.random() < prob
            
        if accept:
            current_state = proposal_state
            current_value = proposal_value
            current_weight = proposal_weight
            
            # Simpan jika ini konfigurasi terbaik yang sah (memenuhi kapasitas)
            if current_value > best_value and current_weight <= capacity:
                best_value = current_value
                best_state = current_state.copy()
                
    return best_state, best_value

# -----------------------------------------------------------------
# CONTOH PENGUJIAN MCMC KNAPSACK
# -----------------------------------------------------------------
print(f"=== HASIL OPTIMASI MCMC KNAPSACK ===")
# Contoh barang audit: berat (misal: jam kerja tim) vs nilai (skor risiko/temuan)
daftar_barang = [
    {'weight': 2, 'value': 3},
    {'weight': 3, 'value': 4},
    {'weight': 4, 'value': 5},
    {'weight': 5, 'value': 8},
    {'weight': 1, 'value': 2}
]
kapasitas_maks = 7

susunan_barang, total_skor = mcmc_knapsack(daftar_barang, capacity=kapasitas_maks, n_iter=50000)
print(f"Kombinasi Barang yang dipilih (0=Tolak, 1=Ambil): {susunan_barang}")
print(f"Total Nilai Optimal                           : {total_skor}")